In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv('Country-data.csv')

print(df.shape)
df.head()

(167, 10)


,country,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp
0,Afghanistan,90.2,10.0,7.58,44.9,1610,9.44,56.2,5.82,553
1,Albania,16.6,28.0,6.55,48.6,9930,4.49,76.3,1.65,4090
2,Algeria,27.3,38.4,4.17,31.4,12900,16.10,76.5,2.89,4460
3,Angola,119.0,62.3,2.85,42.9,5900,22.40,60.1,6.16,3530
4,Antigua and Barbuda,10.3,45.5,6.03,58.9,19100,1.44,76.8,2.13,12200


In [6]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167 entries, 0 to 166
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   country     167 non-null    object 
 1   child_mort  167 non-null    float64
 2   exports     167 non-null    float64
 3   health      167 non-null    float64
 4   imports     167 non-null    float64
 5   income      167 non-null    int64  
 6   inflation   167 non-null    float64
 7   life_expec  167 non-null    float64
 8   total_fer   167 non-null    float64
 9   gdpp        167 non-null    int64  
dtypes: float64(7), int64(2), object(1)
memory usage: 13.2+ KB


,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp
count,167.000000,167.000000,167.000000,167.000000,167.000000,167.000000,167.000000,167.000000,167.000000
mean,38.270060,41.108976,6.815689,46.890215,17144.688623,7.781832,70.555689,2.947964,12964.155689
std,40.328931,27.412010,2.746837,24.209589,19278.067698,10.570704,8.893172,1.513848,18328.704809
min,2.600000,0.109000,1.810000,0.065900,609.000000,-4.210000,32.100000,1.150000,231.000000
25%,8.250000,23.800000,4.920000,30.200000,3355.000000,1.810000,65.300000,1.795000,1330.000000
50%,19.300000,35.000000,6.320000,43.300000,9960.000000,5.390000,73.100000,2.410000,4660.000000
75%,62.100000,51.350000,8.600000,58.750000,22800.000000,10.750000,76.800000,3.880000,14050.000000
max,208.000000,200.000000,17.900000,174.000000,125000.000000,104.000000,82.800000,7.490000,105000.000000


In [9]:
print(df.isnull().sum())
df.duplicated().sum()

country       0
child_mort    0
exports       0
health        0
imports       0
income        0
inflation     0
life_expec    0
total_fer     0
gdpp          0
dtype: int64


0

In [10]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

fig = make_subplots(rows=3, cols=3, subplot_titles=numerical_cols)
for i, col in enumerate(numerical_cols):
    r, c = i // 3 + 1, i % 3 + 1
    fig.add_trace(go.Histogram(x=df[col], nbinsx=30), row=r, col=c)
fig.update_layout(height=750, showlegend=False, title_text="Feature Distributions")
fig.show()

In [11]:
corr = df[numerical_cols].corr()
fig = px.imshow(
    corr, text_auto='.2f', aspect='auto',
    color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
    title='Feature Correlation Heatmap'
)
fig.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

# Step 2a: separate country names (we don't scale strings)
country_names = df['country']
X = df.drop('country', axis=1).copy()

# Step 2b: log-transform skewed features
# np.log1p = log(1+x), safer than log because it handles 0s
skewed_features = ['child_mort', 'income', 'gdpp', 'total_fer'] 
# question why we only did these 4 but not the others?
# is it only determined by eye or how do we do with numbers actually or there is no way a computer choosing those features for us?
for col in skewed_features:
    X[col] = np.log1p(X[col])

# Step 2c: standardize everything
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame for readability
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
print(X_scaled.describe().round(2))

       child_mort  exports  health  imports  income  inflation  life_expec  \
count      167.00   167.00  167.00   167.00  167.00     167.00      167.00   
mean        -0.00     0.00    0.00     0.00    0.00      -0.00        0.00   
std          1.00     1.00    1.00     1.00    1.00       1.00        1.00   
min         -1.70    -1.50   -1.83    -1.94   -2.21      -1.14       -4.34   
25%         -0.83    -0.63   -0.69    -0.69   -0.81      -0.57       -0.59   
50%         -0.11    -0.22   -0.18    -0.15    0.07      -0.23        0.29   
75%          0.94     0.37    0.65     0.49    0.75       0.28        0.70   
max          2.04     5.81    4.05     5.27    2.14       9.13        1.38   

       total_fer    gdpp  
count     167.00  167.00  
mean        0.00   -0.00  
std         1.00    1.00  
min        -1.54   -2.04  
25%        -0.79   -0.87  
50%        -0.23   -0.04  
75%         0.79    0.70  
max         2.36    2.05  


In [13]:
from sklearn.decomposition import PCA

# Step 3a: fit PCA with all components first, to see the variance breakdown
pca_full = PCA()
pca_full.fit(X_scaled)

# explained_variance_ratio_ tells us % of variance each PC captures
explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

print("Variance explained by each component:")
for i, (ev, cv) in enumerate(zip(explained_var, cumulative_var), 1):
    print(f"  PC{i}: {ev:.3f} ({ev*100:.1f}%)   Cumulative: {cv*100:.1f}%")

Variance explained by each component:
  PC1: 0.532 (53.2%)   Cumulative: 53.2%
  PC2: 0.174 (17.4%)   Cumulative: 70.6%
  PC3: 0.133 (13.3%)   Cumulative: 83.9%
  PC4: 0.080 (8.0%)   Cumulative: 91.9%
  PC5: 0.038 (3.8%)   Cumulative: 95.7%
  PC6: 0.024 (2.4%)   Cumulative: 98.1%
  PC7: 0.010 (1.0%)   Cumulative: 99.1%
  PC8: 0.007 (0.7%)   Cumulative: 99.8%
  PC9: 0.002 (0.2%)   Cumulative: 100.0%


In [14]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=[f'PC{i+1}' for i in range(len(explained_var))],
    y=explained_var,
    name='Individual',
    marker_color='steelblue'
))
fig.add_trace(go.Scatter(
    x=[f'PC{i+1}' for i in range(len(cumulative_var))],
    y=cumulative_var,
    name='Cumulative',
    mode='lines+markers',
    line=dict(color='crimson', width=3),
    yaxis='y2'
))
fig.add_hline(y=0.95, line_dash='dash', line_color='gray',
              annotation_text='95% threshold', yref='y2')
fig.update_layout(
    title='PCA Scree Plot',
    xaxis_title='Principal Component',
    yaxis=dict(title='Individual Variance Ratio'),
    yaxis2=dict(title='Cumulative Variance', overlaying='y', side='right', range=[0, 1.05]),
    height=500
)
fig.show()

In [15]:
# Replace N with the number you decided based on the scree plot
N = 5  # e.g., 4 or 5 — depends on what you see
pca = PCA(n_components=N)
X_pca = pca.fit_transform(X_scaled)

X_pca_df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(N)])
print(f"\nReduced from {X_scaled.shape[1]} features to {N} components")
print(f"Variance retained: {pca.explained_variance_ratio_.sum()*100:.1f}%")
X_pca_df.head()


Reduced from 9 features to 5 components
Variance retained: 95.7%


,PC1,PC2,PC3,PC4,PC5
0,-3.408584,0.001364,-0.900100,0.325149,0.263531
1,0.653329,-0.397966,-0.217469,-0.497707,-0.870250
2,-0.149676,-0.512437,1.370548,-0.255664,-0.158876
3,-2.405453,1.060543,1.895645,0.335308,1.180221
4,1.433825,0.226446,-0.006665,-0.538698,-0.061476


In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(N)],
    index=X.columns
)
print("PCA Loadings (how each original feature contributes to each PC):")
print(loadings.round(3))

# Heatmap version for easier reading
fig = px.imshow(
    loadings, text_auto='.2f', aspect='auto',
    color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
    title='PCA Loadings'
)
fig.show()

PCA Loadings (how each original feature contributes to each PC):
              PC1    PC2    PC3    PC4    PC5
child_mort -0.439  0.102  0.036  0.000  0.170
exports     0.246  0.614  0.210  0.100  0.259
health      0.140 -0.192 -0.655  0.670  0.100
imports     0.124  0.715 -0.204  0.177 -0.281
income      0.424 -0.090  0.236  0.028  0.338
inflation  -0.164 -0.100  0.634  0.690 -0.282
life_expec  0.404 -0.168  0.087 -0.075 -0.270
total_fer  -0.402  0.082  0.018  0.130  0.591
gdpp        0.425 -0.108  0.134  0.102  0.445


In [17]:
from sklearn.cluster import KMeans

wcss = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_pca)
    wcss.append(km.inertia_)   # inertia_ IS the WCSS

fig = px.line(
    x=list(K_range), y=wcss, markers=True,
    labels={'x': 'Number of clusters (K)', 'y': 'WCSS (Inertia)'},
    title='Elbow Method'
)
fig.show()

In [19]:
from sklearn.metrics import silhouette_score

sil_scores = []
for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_pca)
    sil_scores.append(silhouette_score(X_pca, labels))

fig = px.line(
    x=list(K_range), y=sil_scores, markers=True,
    labels={'x': 'Number of clusters (K)', 'y': 'Silhouette Score'},
    title='Silhouette Score by K'
)
fig.show()

#print all the shilhouette scores for each K
print("Silhouette Scores for each K:")
for k, score in zip(K_range, sil_scores):
    print(f"  K={k}: {score:.3f}")
    

Silhouette Scores for each K:
  K=2: 0.364
  K=3: 0.262
  K=4: 0.283
  K=5: 0.290
  K=6: 0.269
  K=7: 0.265
  K=8: 0.254
  K=9: 0.236
  K=10: 0.267


In [20]:
optimal_k = 4

kmeans = KMeans(
    n_clusters=optimal_k,
    init='k-means++',
    n_init=10,
    random_state=42
)
cluster_labels = kmeans.fit_predict(X_pca)

# Attach the labels back to your original dataframe so we can interpret them
df_clustered = df.copy()
df_clustered['Cluster'] = cluster_labels

# Quick sanity check — how many countries per cluster?
print("Countries per cluster:")
print(df_clustered['Cluster'].value_counts().sort_index())

Countries per cluster:
Cluster
0     3
1    53
2    44
3    67
Name: count, dtype: int64


In [21]:
viz_df = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'Cluster': cluster_labels.astype(str),  # str so plotly treats it as categorical
    'Country': country_names
})

fig = px.scatter(
    viz_df, x='PC1', y='PC2', color='Cluster',
    hover_data=['Country'],
    title=f'K-Means Clusters (K={optimal_k}) in PC1–PC2 Space',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(marker=dict(size=10, line=dict(width=1, color='white')))
fig.update_layout(height=600)
fig.show()

In [22]:
viz_df_3d = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'PC3': X_pca[:, 2],
    'Cluster': cluster_labels.astype(str),
    'Country': country_names
})

fig = px.scatter_3d(
    viz_df_3d, x='PC1', y='PC2', z='PC3', color='Cluster',
    hover_data=['Country'],
    title='Clusters in 3D PC Space',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(height=700)
fig.show()

In [24]:
# Group by cluster and compute the mean of each original feature
cluster_profile = df_clustered.groupby('Cluster').mean(numeric_only=True).round(2)
print("Average feature values per cluster:")
print(cluster_profile)

Average feature values per cluster:
         child_mort  exports  health  imports    income  inflation  \
Cluster                                                              
0              4.13   176.00    6.79   156.67  64033.33       2.47   
1             85.82    27.02    6.34    43.89   2608.36      10.85   
2              6.02    43.51    9.38    44.38  30768.64       1.82   
3             23.36    44.64    5.50    46.00  17597.01       9.51   

         life_expec  total_fer      gdpp  
Cluster                                   
0             81.43       1.38  57566.67  
1             60.37       4.70   1261.58  
2             78.88       1.70  30640.00  
3             72.66       2.45   8616.27  


In [25]:
# Normalize each column (feature) so highest cluster = 1, lowest = 0
profile_normalized = (cluster_profile - cluster_profile.min()) / (cluster_profile.max() - cluster_profile.min())

fig = px.imshow(
    profile_normalized.T,  # transpose so features are on Y axis, clusters on X
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdYlGn',
    title='Cluster Profile Heatmap (normalized 0–1 across clusters per feature)',
    labels=dict(x='Cluster', y='Feature', color='Relative Level')
)
fig.update_layout(height=500)
fig.show()

In [26]:
fig = px.choropleth(
    df_clustered,
    locations='country',
    locationmode='country names',  # tells plotly to match by country name
    color=df_clustered['Cluster'].astype(str),  # categorical, not continuous
    hover_name='country',
    hover_data=['child_mort', 'income', 'gdpp', 'life_expec'],
    title='Country Clusters (K-Means)',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(height=600)
fig.show()

/tmp/ipykernel_56276/3731647691.py:1: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


In [28]:
# Focus only on the "needs aid" cluster
needy = df_clustered[df_clustered['Cluster'] == 1].copy()

# Build a composite "needs aid" score — higher = needs more help
# Standardize within this group, weight the most relevant features
from sklearn.preprocessing import StandardScaler
features_for_score = ['child_mort', 'income', 'life_expec', 'gdpp', 'total_fer']
scaler_score = StandardScaler()
scaled = scaler_score.fit_transform(needy[features_for_score])

# Bad indicators get +ve weight, good indicators get -ve weight
# Higher composite score = more in need
needy['need_score'] = (
    scaled[:, 0]      # child_mort   (higher = worse)
    - scaled[:, 1]    # income       (higher = better, so subtract)
    - scaled[:, 2]    # life_expec   (higher = better, so subtract)
    - scaled[:, 3]    # gdpp         (higher = better, so subtract)
    + scaled[:, 4]    # total_fer    (higher fertility correlated with underdevelopment)
)

top_priority = needy.sort_values('need_score', ascending=False).head(15)
print("Top 15 countries most in need of aid:")
print(top_priority[['country', 'child_mort', 'income', 'gdpp', 'life_expec', 'need_score']])

Top 15 countries most in need of aid:
                      country  child_mort  income  gdpp  life_expec  \
66                      Haiti       208.0    1500   662        32.1   
31   Central African Republic       149.0     888   446        47.5   
112                     Niger       123.0     814   348        58.8   
132              Sierra Leone       160.0    1220   399        55.0   
32                       Chad       150.0    1930   897        56.5   
37           Congo, Dem. Rep.       116.0     609   334        57.5   
97                       Mali       137.0    1870   708        59.5   
26                    Burundi        93.6     764   231        57.7   
106                Mozambique       101.0     918   419        54.5   
25               Burkina Faso       116.0    1430   575        57.9   
94                     Malawi        90.5    1030   459        53.1   
64              Guinea-Bissau       114.0    1390   547        55.6   
63                     Guinea       109